In [4]:
import os
import datetime
import pypff
import html
from bs4 import BeautifulSoup
import re
import json

def extract_messages(pst_file):
    pst = pypff.file()
    pst.open(pst_file)
    root = pst.get_root_folder()
    
    messages = []
    message_id = 1  # Initialize message ID counter
    
    def parse_folder(folder):
        nonlocal message_id  # Use the nonlocal keyword to modify the outer message_id
        for message in folder.sub_messages:
            delivery_time = message.get_delivery_time()
            if delivery_time:
                delivery_time = datetime.datetime.fromtimestamp(delivery_time.timestamp()).strftime('%Y-%m-%d %H:%M:%S')
            
            subject = message.subject or "No Subject"
            sender = message.sender_name or "Unknown Sender"
            
            # Attempt to get HTML body, fallback to plain text or RTF if necessary
            body = message.get_html_body()
            if body:
                try:
                    body = body.decode('utf-8')  # Try decoding with UTF-8
                except UnicodeDecodeError:
                    body = body.decode('iso-8859-1')  # Fallback to ISO-8859-1
            else:
                body = message.get_plain_text_body() or message.get_rtf_body() or "No content"
            
            # Parse HTML content into plain text with custom spacing
            soup = BeautifulSoup(body, 'html.parser')

            # Replace specific tags with meaningful spacing
            for br in soup.find_all("br"):
                br.replace_with("\n")
            for p in soup.find_all("p"):
                p.insert_before("\n")
            for li in soup.find_all("li"):
                li.insert_before("\n- ")

            plain_text_body = soup.get_text()

            # Clean up excess newlines
            plain_text_body = '\n'.join([line.strip() for line in plain_text_body.splitlines() if line.strip()])

            # Check if the body contains "israel" or "war bonds"
            if re.search(r'israel|war bonds', plain_text_body, re.IGNORECASE):
                messages.append({
                    'id': message_id,
                    'subject': subject,
                    'sender': sender,
                    'datetime': delivery_time or "Unknown Date",
                    'page_content': f"Subject: {subject}\nFrom: {sender}\nDate: {delivery_time or 'Unknown Date'}\nBody:\n{plain_text_body}"
                })
                message_id += 1  # Increment the message ID for the next message
        
        for sub_folder in folder.sub_folders:
            parse_folder(sub_folder)
    
    parse_folder(root)
    pst.close()
    return messages

def output_to_json_file(messages, output_file):
    json_data = {
        "messages": messages
    }
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)

def main():
    input_file = "../data/input/emails.pst"
    output_file = "../data/output/emails.json"
    
    if not os.path.exists(input_file):
        print(f"Error: Input file {input_file} not found.")
        return
    
    try:
        print("Extracting messages from PST file...")
        messages = extract_messages(input_file)
        
        print(f"Writing filtered messages to JSON file: {output_file}")
        output_to_json_file(messages, output_file)
        
        print(f"Extraction and output completed successfully. {len(messages)} relevant messages found.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == "__main__":
    main()

Extracting messages from PST file...
Writing filtered messages to JSON file: ../data/output/emails.json
Extraction and output completed successfully. 50 relevant messages found.
